# Classify Tree Species at CLBJ with Random Forest and Field Data in the Earth Engine Python API

This tutorial classifies woody plant species and three land cover types (grassland, water, and shade) at the NEON Lyndon B. Johnson National Grassland (CLBJ, domain D11) site using 2022 NEON Airborne Observation Platform (AOP) hyperspectral reflectance and lidar canopy height data. It demonstrates how to build training data from NEON vegetation structure field observations and train a random forest classifier in Google Earth Engine (GEE).

This is Phase 3 of the tutorial series. It improves on Phase 2 by testing the classifier on field plots that it did not see during training:
- **Leave-one-plot-out cross-validation** to measure how well the classifier generalizes to new locations. Stems in the same plot have similar spectra (spatial autocorrelation), so a random split of stems gives an accuracy that is too high.
- 2022 AOP data from the BRDF-corrected reflectance collection (`HSI_REFL/002`)
- Training data built directly from NEON vegetation structure (VST) field data (DP1.10098.001), collected from April through November 2022
- A clear account of the limits of the training data, so that you can judge how far to trust the accuracy numbers

## Background Tutorials

This tutorial builds on methods and tools from the following NEON tutorials:

* [Using neonDataStackR to access NEON data](https://www.neonscience.org/resources/learning-hub/tutorials/neondatastackr)
* [Work with NEON AOP reflectance data in Python with geemap](https://www.neonscience.org/resources/learning-hub/tutorials/aop-refl-py-geemap)
* [Classify tree species with NEON AOP data and random forest classification in Google Earth Engine](https://www.neonscience.org/resources/learning-hub/tutorials/aop-gee-random-forest-classification)

## Learning Objectives

After completing this tutorial, you will be able to:

* Download and process NEON vegetation structure data using the NEON API
* Compute stem positions from plot-based measurements using the geoNEON formula
* Generate non-tree training classes using manual selection methods
* Load and preprocess NEON AOP reflectance and CHM data in Google Earth Engine
* Evaluate a classifier with leave-one-plot-out cross-validation to avoid inflated accuracy from spatial autocorrelation
* Train and evaluate a random forest classifier for tree species mapping
* Interpret cross-validated accuracy, the confusion matrix, and per-class producer's and user's accuracy

## Data Products and Study Site

The notebook uses the following NEON data products. Earth Engine hosts the AOP products as image collections and the site boundaries as feature collections.

| NEON data product | ID | Earth Engine asset |
|---|---|---|
| Spectrometer orthorectified surface directional reflectance - mosaic (BRDF-corrected) | [DP3.30006.002](https://data.neonscience.org/data-products/DP3.30006.002) | `projects/neon-prod-earthengine/assets/HSI_REFL/002/2022_CLBJ_6` |
| Ecosystem structure (canopy height model, CHM) | [DP3.30015.001](https://data.neonscience.org/data-products/DP3.30015.001) | `projects/neon-prod-earthengine/assets/CHM/001` |
| Woody plant vegetation structure | [DP1.10098.001](https://data.neonscience.org/data-products/DP1.10098.001) | Downloaded via NEON API |
| Tower airshed | | `projects/neon-prod-earthengine/assets/Feature_Collections/90percent_footprint` |
| Terrestrial Observation System boundary | | `projects/neon-prod-earthengine/assets/Feature_Collections/terrestrialSamplingBoundaries` |

**Site:** CLBJ, Lyndon B. Johnson National Grassland, NEON domain D11 (Texas).

**Year:** 2022. The AOP flight was in late spring. We use VST data from April through November 2022. The April and May records in the apparent individual table cover only two plots, which is too few for plot-level validation. The November bout adds more plots. Live/dead status and canopy position rarely change within a few months, but this gap between the flight and some field measurements is a limitation that we discuss at the end.

**Reference data:** The training points are stem locations from VST field measurements, converted to geographic coordinates using the geoNEON formula. We filter to live trees that are at least 1 m tall and are in full sun, partially shaded, or mostly shaded positions. Non-tree classes (grassland, water, shade) are generated using threshold-based rules on NDVI, brightness, and CHM.

## Setup

Import the packages and initialize Earth Engine. Earth Engine requires a Google Cloud project for every session.

In [1]:
import functools
import os
import math

import ee
import geemap
import neonutilities as nu
import pandas as pd
import numpy as np
import pyproj
import requests

# Change this to the ID of your own Google Cloud project that is registered for Earth Engine.
PROJECT_ID = 'neon-sandbox-dataflow-ee'

ee.Initialize(project=PROJECT_ID)

## Workflow

1. Load boundaries and imagery from Earth Engine
2. Download and process VST field data to create tree species training points
3. Generate non-tree classes using threshold-based methods
4. Assign each field plot to a cross-validation fold and extract the predictor values
5. Train a random forest classifier and evaluate it with leave-one-plot-out cross-validation

## Step 1: Load Boundaries and Imagery

First, load the site boundaries and 2022 AOP data from Earth Engine. The airshed defines the study area, and the TOS boundary shows the terrestrial sampling extent.

In [2]:
# Specify the CLBJ location (longitude, latitude)
geo = ee.Geometry.Point([-97.5706464, 33.4045729])

# Tower airshed boundary (90% footprint) - note capital SiteID
airshed_collection = ee.FeatureCollection('projects/neon-prod-earthengine/assets/Feature_Collections/90percent_footprint')
clbj_airshed = airshed_collection.filter(ee.Filter.eq('SiteID', 'CLBJ'))

# Union the two polygons into one boundary
CLBJ_Airshed = clbj_airshed.union()

# Terrestrial Observation System (TOS) boundary - note lowercase siteID
tos_collection = ee.FeatureCollection('projects/neon-prod-earthengine/assets/Feature_Collections/terrestrialSamplingBoundaries')
CLBJ_TOS = tos_collection.filter(ee.Filter.eq('siteID', 'CLBJ'))

### Canopy Height Model (CHM)

Load the 2022 CHM and mask negative values (no-data).

In [3]:
# Load CHM for 2022 at CLBJ
chm_collection = ee.ImageCollection('projects/neon-prod-earthengine/assets/CHM/001')
CLBJ_CHM2022 = (chm_collection
                .filterDate('2022-01-01', '2022-12-31')
                .filterBounds(geo)
                .first())

# The band is named 'CHM'. Mask values < 0 (no-data).
CLBJ_CHM2022mask = CLBJ_CHM2022.select('CHM').updateMask(CLBJ_CHM2022.select('CHM').gte(0))

print('CHM band:', CLBJ_CHM2022mask.bandNames().getInfo())

CHM band: ['CHM']


### Surface Directional Reflectance (BRDF-corrected)

Load the 2022 BRDF-corrected reflectance mosaic. The image has 426 bands (B001-B426) stored as integers scaled by a factor of 10,000. We divide by the scaling factor to get fractional reflectance (0-1).

Water vapor absorbs most light in certain wavelength ranges, making those bands noisy. We remove:
- Bands 195-205 (water absorption)
- Bands 287-310 (water absorption)
- Bands 416-425 (noisy edge bands)

This leaves 381 bands for the classification.

In [4]:
# Load 2022 reflectance
CLBJ_SDR2022 = ee.Image('projects/neon-prod-earthengine/assets/HSI_REFL/002/2022_CLBJ_6')

# Select bands B001-B426 and apply the scaling factor
band_names_426 = [f'B{i:03d}' for i in range(1, 427)]
CLBJ_SDR2022_bands = CLBJ_SDR2022.select(band_names_426)

# Get the SCALING_FACTOR property and divide
scaling_factor = ee.Number(CLBJ_SDR2022.get('SCALING_FACTOR'))
CLBJ_SDR2022_scaled = CLBJ_SDR2022_bands.divide(scaling_factor)

print('Scaling factor:', scaling_factor.getInfo())

Scaling factor: 10000


In [5]:
# Remove water absorption and noisy bands
bandsToRemove = [f'B{i:03d}' for i in [*range(195, 206), *range(287, 311), *range(416, 426)]]
all_band_names = ee.List([f'B{i:03d}' for i in range(1, 427)])
bandsToKeep = all_band_names.removeAll(bandsToRemove)

CLBJ_SDR2022subset = CLBJ_SDR2022_scaled.select(bandsToKeep)

print('Bands removed:', len(bandsToRemove))
print('Bands kept:', bandsToKeep.size().getInfo())  # 381 expected

Bands removed: 45
Bands kept: 381


In [6]:
# Display version: select RGB bands and mask no-data
# For 2022 data, use bands B053 (red), B035 (green), B019 (blue)
CLBJ_SDR2022mask = CLBJ_SDR2022_scaled.select(['B053', 'B035', 'B019'])

# Mask negative values (no-data indicators)
CLBJ_SDR2022mask = CLBJ_SDR2022mask.updateMask(CLBJ_SDR2022mask.select('B053').gte(0))

### Create the Predictor Composite

Combine the 381 reflectance bands and the CHM into a single composite image. This composite provides all the predictor variables for the random forest classifier.

In [7]:
# Combine the 381 reflectance bands and 1 CHM band (382 bands total)
CLBJ_composite = CLBJ_SDR2022subset.addBands(CLBJ_CHM2022mask)

# Clip to the airshed boundary
CLBJ_composite_airshed = CLBJ_composite.clip(CLBJ_Airshed)

print('Composite bands:', CLBJ_composite_airshed.bandNames().size().getInfo())  # 382 expected

Composite bands: 382


### Map 1: Input Data

Display the reflectance RGB composite, CHM, and boundaries on an interactive map.

In [8]:
map_inputs = geemap.Map()

# RGB composite (adjust min/max for 2022 data - fractional reflectance)
map_inputs.addLayer(CLBJ_SDR2022mask, {'min': 0.005, 'max': 0.10}, 'CLBJ SDR 2022 RGB', False)
map_inputs.addLayer(CLBJ_CHM2022mask, {'min': 0, 'max': 33, 'palette': ['green', 'yellow', 'red']}, 'CLBJ CHM 2022', False)
map_inputs.addLayer(CLBJ_composite_airshed,
                    {'bands': ['B053', 'B035', 'B019'], 'min': 0.005, 'max': 0.10},
                    'CLBJ-Airshed Composite 2022', True)

# Boundaries with transparent fill
map_inputs.addLayer(CLBJ_TOS.style(width=3, color='blue', fillColor='#00000000'), {}, 'CLBJ TOS', False)
map_inputs.addLayer(CLBJ_Airshed.style(width=3, color='white', fillColor='#00000000'), {}, 'CLBJ Airshed', False)

map_inputs.centerObject(geo, 16)
map_inputs

Map(center=[33.404572900000005, -97.5706464], controls=(WidgetControl(options=['position', 'transparent_bg'], …

## Step 2: Download and Process VST Field Data

We'll download NEON vegetation structure (VST) data from April and May 2022 to build our tree species training points. The VST data includes stem measurements, species identifications, and plot-relative positions that we'll convert to geographic coordinates.

### Download VST Data

The NEON API requires a token for data downloads. Set your token as an environment variable `NEON_TOKEN` before running this notebook. You can obtain a token from your NEON user account.

In [9]:
# Read NEON API token from environment variable
neon_token = os.environ.get('NEON_TOKEN')
if not neon_token:
    raise ValueError('NEON_TOKEN environment variable is not set. '
                     'Please set it to your NEON API token before running this notebook.')

print('NEON token found (not displayed for security)')

NEON token found (not displayed for security)


In [10]:
# Download VST data for CLBJ, April-November 2022
# The April and May records of the apparent individual table cover only 2 plots.
# The November 2022 bout adds more plots. This may take a minute or two.
vst_data = nu.load_by_product(
    dpid='DP1.10098.001',
    site='CLBJ',
    startdate='2022-04',
    enddate='2022-12',
    package='basic',
    check_size=False,
    token=neon_token
)

print('VST data downloaded')

Finding available files
100%|██████████| 3/3 [00:01<00:00,  1.53it/s]
Stacking data files
100%|██████████| 5/5 [00:00<00:00, 10.36it/s]


VST data downloaded


### Extract VST Tables

The VST data product includes several tables. We need:
- `vst_mappingandtagging`: plot-relative stem positions and identifiers
- `vst_apparentindividual`: plant status, canopy position, and height

**Choosing the date range.** Stem positions and species (`vst_mappingandtagging`) do not change between bouts, but the attributes that we use to filter stems (`plantStatus`, `canopyPosition`, and `height`) come from `vst_apparentindividual`, which is measured at each bout. When we download only April and May 2022, the apparent individual table covers just two plots (`CLBJ_036` and `CLBJ_043`), and the stems that pass our filters are all in `CLBJ_043`. One plot cannot be split into training and test plots. We therefore download April through November 2022, which adds the November 2022 bout and more plots. The cost is that some attributes were measured up to six months after the AOP flight.


In [11]:
# Extract the relevant tables
mapping_tagging = vst_data['vst_mappingandtagging']
apparent_individual = vst_data['vst_apparentindividual']

print(f'Mapping and tagging records: {len(mapping_tagging)}')
print(f'Apparent individual records: {len(apparent_individual)}')

Mapping and tagging records: 3589
Apparent individual records: 412


In [12]:
# Display key columns from apparent individual table
individual_cols = ['individualID', 'plantStatus', 'canopyPosition', 'height']
print('Apparent Individual Table (sample rows):')
print(f'Shows plant characteristics: status, canopy position, and height\n')
display(apparent_individual[individual_cols].head(10))

print(f'\nCanopy positions present:')
print(apparent_individual['canopyPosition'].value_counts())
print(f'\nHeight range: {apparent_individual["height"].min():.1f} - {apparent_individual["height"].max():.1f} m')

Apparent Individual Table (sample rows):
Shows plant characteristics: status, canopy position, and height



,individualID,plantStatus,canopyPosition,height
0,NEON.PLA.D11.CLBJ.02152,"Live, broken bole",Partially shaded,3.4
1,NEON.PLA.D11.CLBJ.03192,"Live, physically damaged",Partially shaded,0.8
2,NEON.PLA.D11.CLBJ.03274,"Live, physically damaged",,NaN
3,NEON.PLA.D11.CLBJ.03265,"Live, physically damaged",,NaN
4,NEON.PLA.D11.CLBJ.03284,"Live, physically damaged",,NaN
5,NEON.PLA.D11.CLBJ.01011,"Live, physically damaged",,NaN
6,NEON.PLA.D11.CLBJ.01012,"Live, physically damaged",,NaN
7,NEON.PLA.D11.CLBJ.02176,Live,Partially shaded,8.0
8,NEON.PLA.D11.CLBJ.03290,"Live, physically damaged",Full sun,1.0
9,NEON.PLA.D11.CLBJ.03378,Live,Mostly shaded,1.1



Canopy positions present:
canopyPosition
                    123
Mostly shaded        97
Partially shaded     96
Full shade           52
Full sun             24
Open grown           20
Name: count, dtype: int64

Height range: 0.3 - 12.6 m


In [13]:
# Display key columns from mapping and tagging table
mapping_cols = ['plotID', 'pointID', 'individualID', 'taxonID', 'stemDistance', 'stemAzimuth']
print('Mapping and Tagging Table (sample rows):')
print(f'Shows plot locations and stem positions relative to plot corner points\n')
display(mapping_tagging[mapping_cols].head(10))

print(f'\nMissing coordinates: {mapping_tagging["stemDistance"].isna().sum()} of {len(mapping_tagging)} records ({100*mapping_tagging["stemDistance"].isna().sum()/len(mapping_tagging):.1f}%)')
print(f'Species present: {mapping_tagging["taxonID"].nunique()} unique taxa')

Mapping and Tagging Table (sample rows):
Shows plot locations and stem positions relative to plot corner points



,plotID,pointID,individualID,taxonID,stemDistance,stemAzimuth
0,CLBJ_029,,NEON.PLA.D11.CLBJ.00755,ULCR,NaN,NaN
1,CLBJ_029,49,NEON.PLA.D11.CLBJ.00752,QUERC,6.7,167.2
2,CLBJ_029,49,NEON.PLA.D11.CLBJ.00762,ULCR,8.5,155.7
3,CLBJ_029,,NEON.PLA.D11.CLBJ.00754,ULCR,NaN,NaN
4,CLBJ_029,,NEON.PLA.D11.CLBJ.00769,ULCR,NaN,NaN
5,CLBJ_029,,NEON.PLA.D11.CLBJ.00753,ULCR,NaN,NaN
6,CLBJ_029,,NEON.PLA.D11.CLBJ.00782,CELA,NaN,NaN
7,CLBJ_029,,NEON.PLA.D11.CLBJ.00760,ULMUS,NaN,NaN
8,CLBJ_029,,NEON.PLA.D11.CLBJ.00763,ULCR,NaN,NaN
9,CLBJ_029,,NEON.PLA.D11.CLBJ.00768,QUERC,NaN,NaN



Missing coordinates: 2757 of 3589 records (76.8%)
Species present: 40 unique taxa


### Examine VST Table Structure

Before processing the VST data, let's examine the table structure to understand what columns are available and what the data looks like. This helps us understand what we're working with when we filter and process the data.

### Compute Stem Geographic Coordinates

VST data record stem positions relative to plot corner points using `stemDistance` and `stemAzimuth`. We use the geoNEON formula to convert these to absolute coordinates:

1. Query the NEON location API for each plot point's UTM coordinates
2. Apply the geoNEON formula: 
   - `adjEasting = pointEasting + stemDistance × sin(stemAzimuth × π/180)`
   - `adjNorthing = pointNorthing + stemDistance × cos(stemAzimuth × π/180)`
3. Convert from UTM Zone 14N (EPSG:32614) to WGS84 lon/lat (EPSG:4326)

In [14]:
# Get unique plot/point combinations
plot_points = mapping_tagging[['plotID', 'pointID']].drop_duplicates()
print(f'Unique plot-point combinations: {len(plot_points)}')

Unique plot-point combinations: 214


In [15]:
# Function to query NEON location API for point coordinates
def get_point_location(plot_id, point_id, cache={}):
    """Get UTM coordinates for a plot point from the NEON location API.
    
    Args:
        plot_id: Plot identifier (e.g., 'CLBJ_001')
        point_id: Point identifier (e.g., '41')
        cache: Dictionary to cache results
    
    Returns:
        Tuple of (easting, northing) or (None, None) if not found
    """
    key = (plot_id, point_id)
    if key in cache:
        return cache[key]
    
    # Construct the named location
    named_location = f"{plot_id}.basePlot.vst.{point_id}"
    url = f"https://data.neonscience.org/api/v0/locations/{named_location}"
    
    try:
        response = requests.get(url, timeout=10)
        if response.status_code == 200:
            data = response.json()
            easting = data['data']['locationUtmEasting']
            northing = data['data']['locationUtmNorthing']
            cache[key] = (easting, northing)
            return (easting, northing)
        else:
            cache[key] = (None, None)
            return (None, None)
    except Exception as e:
        print(f"Error fetching {named_location}: {e}")
        cache[key] = (None, None)
        return (None, None)

print('Location API query function defined')

Location API query function defined


In [16]:
# Query point locations (this may take a few minutes due to API rate limits)
print('Querying NEON location API for plot point coordinates...')
location_cache = {}

for n, (_, row) in enumerate(plot_points.iterrows(), start=1):
    get_point_location(row['plotID'], row['pointID'], location_cache)
    if n % 25 == 0:
        print(f'  Queried {n}/{len(plot_points)} locations')

print(f'Location query complete: {len(location_cache)} unique locations cached')

Querying NEON location API for plot point coordinates...
  Queried 25/214 locations
  Queried 50/214 locations
  Queried 75/214 locations
  Queried 100/214 locations
  Queried 125/214 locations
  Queried 150/214 locations
  Queried 175/214 locations
  Queried 200/214 locations
Location query complete: 214 unique locations cached


In [17]:
# Apply geoNEON formula to compute stem positions
def compute_stem_position(row):
    """Compute stem UTM position using the geoNEON formula."""
    point_easting, point_northing = location_cache.get(
        (row['plotID'], row['pointID']),
        (None, None)
    )
    
    if point_easting is None or pd.isna(row['stemDistance']) or pd.isna(row['stemAzimuth']):
        return pd.Series({'stemEasting': None, 'stemNorthing': None})
    
    # geoNEON formula
    azimuth_rad = row['stemAzimuth'] * math.pi / 180
    stem_easting = point_easting + row['stemDistance'] * math.sin(azimuth_rad)
    stem_northing = point_northing + row['stemDistance'] * math.cos(azimuth_rad)
    
    return pd.Series({'stemEasting': stem_easting, 'stemNorthing': stem_northing})

# Apply to mapping_tagging table
mapping_tagging[['stemEasting', 'stemNorthing']] = mapping_tagging.apply(
    compute_stem_position,
    axis=1
)

print(f'Stem positions computed for {mapping_tagging["stemEasting"].notna().sum()} stems')

Stem positions computed for 830 stems


In [18]:
# Convert UTM Zone 14N to WGS84 lon/lat
transformer = pyproj.Transformer.from_crs('EPSG:32614', 'EPSG:4326', always_xy=True)

def utm_to_lonlat(row):
    """Convert UTM coordinates to lon/lat."""
    if pd.isna(row['stemEasting']) or pd.isna(row['stemNorthing']):
        return pd.Series({'longitude': None, 'latitude': None})
    
    lon, lat = transformer.transform(row['stemEasting'], row['stemNorthing'])
    return pd.Series({'longitude': lon, 'latitude': lat})

mapping_tagging[['longitude', 'latitude']] = mapping_tagging.apply(utm_to_lonlat, axis=1)

print(f'Geographic coordinates computed for {mapping_tagging["longitude"].notna().sum()} stems')

Geographic coordinates computed for 830 stems


### Join Tables and Filter to Visible Trees

Join the mapping and tagging table with the apparent individual table to get the plant status, canopy position, and height of each stem. Some individuals were measured in more than one bout, so the apparent individual table can hold several records for one individual. We keep the most recent record of each individual, so that each stem appears once.

Then filter to:
- Live trees (`plantStatus` contains "Live")
- Canopy position of "Full sun", "Partially shaded", or "Mostly shaded"
- Height ≥ 1 m
- Valid `stemDistance` and `stemAzimuth`, so that we can compute a position

**Why relaxed filters?** A strict filter (full sun or open grown, height ≥ 3 m) keeps only trees that the sensor sees well, but it leaves very few stems at CLBJ, because most of the measured trees are short or shaded (see the canopy position counts above). The relaxed filter keeps more stems. The cost is that some of them are understory stems that the sensor may not see, which adds noise to the training data.

In [19]:
# Keep the most recent apparent individual record for each individual
apparent_latest = (
    apparent_individual
    .sort_values('date')
    .drop_duplicates('individualID', keep='last')
    [['individualID', 'plantStatus', 'canopyPosition', 'height']]
)

# Join tables on individualID
vst_joined = mapping_tagging.merge(apparent_latest, on='individualID', how='left')

print(f'Apparent individual records: {len(apparent_individual)} ({len(apparent_latest)} individuals)')
print(f'Joined table: {len(vst_joined)} records')

Apparent individual records: 412 (345 individuals)
Joined table: 3589 records


In [20]:
# Relaxed filters for 2022 CLBJ data
vst_filtered = vst_joined[
    (vst_joined['stemDistance'].notna()) &
    (vst_joined['stemAzimuth'].notna()) &
    (vst_joined['longitude'].notna()) &
    (vst_joined['plantStatus'].str.contains('Live', na=False)) &
    (vst_joined['canopyPosition'].isin(['Full sun', 'Partially shaded', 'Mostly shaded'])) &
    (vst_joined['height'] >= 1.0)
].drop_duplicates('individualID').copy()

print(f"Filtered trees: {len(vst_filtered)} stems in {vst_filtered['plotID'].nunique()} plots")
print(f"\nStems by species and plot:")
display(vst_filtered.groupby(['taxonID', 'plotID']).size().unstack(fill_value=0))

Filtered trees: 65 stems in 5 plots

Stems by species and plot:


plotID,CLBJ_002,CLBJ_003,CLBJ_004,CLBJ_006,CLBJ_043
taxonID,,,,,
JUVI,5,0,0,0,4
PRME,0,1,0,0,0
QUMA3,0,1,0,4,1
QUST,8,13,7,4,9
ULAL,1,0,0,0,0
ULCR,4,0,0,0,3


### Keep Species That Occur in More Than One Plot

Leave-one-plot-out cross-validation holds out one plot at a time. If a species occurs in only one plot, then the classifier never sees that species when the plot is held out, and the species can never be predicted correctly. A species with very few stems also gives unstable accuracy numbers. We therefore keep only species with at least 5 stems in at least 2 plots, and we drop the other species from the training data.

In [21]:
# Keep species with at least 5 stems in at least 2 plots
species_stats = vst_filtered.groupby('taxonID').agg(
    stems=('individualID', 'size'),
    plots=('plotID', 'nunique')
)
print('Stems and plots for each species:')
display(species_stats)

keep_species = species_stats[(species_stats['stems'] >= 5) & (species_stats['plots'] >= 2)].index
dropped_species = sorted(set(species_stats.index) - set(keep_species))
print(f'\nDropped species (fewer than 5 stems or fewer than 2 plots): {dropped_species}')

vst_filtered = vst_filtered[vst_filtered['taxonID'].isin(keep_species)].copy()
print(f"Final tree samples: {len(vst_filtered)} stems in {vst_filtered['plotID'].nunique()} plots")

Stems and plots for each species:


,stems,plots
taxonID,,
JUVI,9,2
PRME,1,1
QUMA3,6,3
QUST,41,5
ULAL,1,1
ULCR,7,2



Dropped species (fewer than 5 stems or fewer than 2 plots): ['PRME', 'ULAL']
Final tree samples: 63 stems in 5 plots


## Step 3: Generate Non-Tree Classes

VST data only includes trees. We need to add training points for three additional land cover classes:
- **GRSS (Grassland)**: Low vegetation with high greenness
- **WATR (Water)**: Very low NDVI and low brightness
- **SHADE (Shade)**: Dark areas under tall canopy

We'll use threshold-based rules on NDVI, brightness, and CHM to identify these areas.

### Compute NDVI and Brightness

In [22]:
# Compute NDVI from scaled reflectance
# NDVI = (NIR - Red) / (NIR + Red)
# B090 is NIR (~800nm), B054 is Red (~670nm)
nir = CLBJ_SDR2022_scaled.select('B090')
red = CLBJ_SDR2022_scaled.select('B054')
ndvi = nir.subtract(red).divide(nir.add(red)).rename('NDVI')

# Compute brightness as mean of visible bands (B019-B060)
visible_bands = [f'B{i:03d}' for i in range(19, 61)]
brightness = CLBJ_SDR2022_scaled.select(visible_bands).reduce(ee.Reducer.mean()).rename('brightness')

print('NDVI and brightness computed')

NDVI and brightness computed


### Define Threshold Rules

Create masks for each non-tree class based on spectral and structural characteristics.

In [23]:
# Get CHM for threshold calculations
chm = CLBJ_CHM2022mask.select('CHM')

# GRSS (grassland): high NDVI, low-medium height
grss_mask = chm.lt(4.0).And(ndvi.gt(0.4))

# WATR (water): low NDVI, low brightness
watr_mask = ndvi.lt(0.3).And(brightness.lt(0.03))

# SHADE (shadow): very low brightness
shade_mask = brightness.lt(0.025)

print('Threshold masks created')

Threshold masks created


In [24]:
# Sample points from each mask (10-15 points per class)
# Use stratifiedSample with a dummy class property

# Create single-band images for sampling
grss_image = grss_mask.selfMask().clip(CLBJ_Airshed)
watr_image = watr_mask.selfMask().clip(CLBJ_Airshed)
shade_image = shade_mask.selfMask().clip(CLBJ_Airshed)

# Sample points
grss_points = grss_image.sample(
    region=CLBJ_Airshed,
    scale=1,
    numPixels=12,
    seed=42,
    geometries=True
)

watr_points = watr_image.sample(
    region=CLBJ_Airshed,
    scale=1,
    numPixels=12,
    seed=43,
    geometries=True
)

shade_points = shade_image.sample(
    region=CLBJ_Airshed,
    scale=1,
    numPixels=12,
    seed=44,
    geometries=True
)

print('Non-tree points sampled')
print(f'  GRSS: {grss_points.size().getInfo()} points')
print(f'  WATR: {watr_points.size().getInfo()} points')
print(f'  SHADE: {shade_points.size().getInfo()} points')

Non-tree points sampled
  GRSS: 2 points
  WATR: 0 points
  SHADE: 2 points


In [25]:
# Manually selected non-tree training points (from visual inspection of 2022 image)

# Water points (from ponds/lakes in the airshed)
watr_coords = [
    [-97.57458, 33.38499],
    [-97.57383, 33.38580],
    [-97.57270, 33.38593],
    [-97.57339, 33.38527],
    [-97.57251, 33.38495],
    [-97.57374, 33.38454],
    [-97.57040, 33.40716],
    [-97.56965, 33.40669],
    [-97.56856, 33.40553],
    [-97.57010, 33.40651]
]

# Grassland points (open areas, low vegetation)
grss_coords = [
    [-97.57794, 33.40790],
    [-97.57343, 33.41121],
    [-97.54877, 33.41231],
    [-97.55444, 33.41417],
    [-97.61723, 33.41045],
    [-97.60071, 33.41111],
    [-97.65015, 33.41466],
    [-97.62499, 33.41824],
    [-97.61812, 33.41852],
    [-97.61755, 33.41786]
]

# Shade points (shadows under dense canopy)
shade_coords = [
    [-97.62058, 33.41911],
    [-97.62008, 33.41934],
    [-97.62239, 33.41942],
    [-97.62245, 33.41968],
    [-97.61363, 33.42193],
    [-97.60779, 33.41329],
    [-97.60730, 33.41298],
    [-97.60689, 33.41319],
    [-97.60876, 33.41297],
    [-97.60916, 33.41316]
]

# Create FeatureCollections with class labels
grss_points = ee.FeatureCollection([
    ee.Feature(ee.Geometry.Point(c), {'class': 7, 'className': 'GRSS'}) 
    for c in grss_coords
])

watr_points = ee.FeatureCollection([
    ee.Feature(ee.Geometry.Point(c), {'class': 8, 'className': 'WATR'}) 
    for c in watr_coords
])

shade_points = ee.FeatureCollection([
    ee.Feature(ee.Geometry.Point(c), {'class': 9, 'className': 'SHADE'}) 
    for c in shade_coords
])

print('Non-tree training points created:')
print(f'  GRSS: {len(grss_coords)} points')
print(f'  WATR: {len(watr_coords)} points')
print(f'  SHADE: {len(shade_coords)} points')
print(f'  Total non-tree: {len(grss_coords) + len(watr_coords) + len(shade_coords)} points')

Non-tree training points created:
  GRSS: 10 points
  WATR: 10 points
  SHADE: 10 points
  Total non-tree: 30 points


In [26]:
# Create map to visualize all reference points
points_map = geemap.Map()

# Add RGB basemap
points_map.addLayer(
    CLBJ_SDR2022subset,
    {'bands': ['B053', 'B035', 'B019'], 'min': 0.005, 'max': 0.10},
    'RGB 2022',
    True
)

# Add non-tree points with distinct colors
points_map.addLayer(
    grss_points.style(pointSize=5, color='00FF00'),  # Bright green
    {},
    'GRSS (grassland)',
    True
)

points_map.addLayer(
    watr_points.style(pointSize=5, color='0000FF'),  # Blue
    {},
    'WATR (water)',
    True
)

points_map.addLayer(
    shade_points.style(pointSize=5, color='000000'),  # Black
    {},
    'SHADE (shadow)',
    True
)

# Add airshed boundary
points_map.addLayer(
    CLBJ_Airshed.style(fillColor='00000000', color='FFFFFF', width=2),
    {},
    'Airshed',
    True
)

points_map.centerObject(CLBJ_Airshed, 13)
points_map

Map(center=[33.40123000138371, -97.57000034548355], controls=(WidgetControl(options=['position', 'transparent_…

### Visualize Threshold Masks

Display the threshold masks on a map to verify they capture the intended land cover types.

In [ ]:
map_thresholds = geemap.Map()

# Base layer
map_thresholds.addLayer(CLBJ_SDR2022mask, {'min': 0.005, 'max': 0.10}, 'CLBJ SDR 2022 RGB', True)

# Threshold masks
map_thresholds.addLayer(grss_mask.selfMask(), {'palette': ['lightgreen']}, 'GRSS mask', True)
map_thresholds.addLayer(watr_mask.selfMask(), {'palette': ['blue']}, 'WATR mask', True)
map_thresholds.addLayer(shade_mask.selfMask(), {'palette': ['black']}, 'SHADE mask', True)

map_thresholds.centerObject(geo, 16)
map_thresholds

### Build Complete Training FeatureCollection

Combine the tree species points from VST with the non-tree points. Each point gets three properties that the rest of the notebook uses:
- `class`: the numeric class label. Tree species get the labels 0, 1, 2, ... in alphabetical order. The non-tree classes keep the labels 7 (GRSS), 8 (WATR), and 9 (SHADE). The labels 4–6 are not used, so the classified map must show 10 values.
- `plotID`: the VST plot of a tree stem
- `fold`: the cross-validation fold. For tree stems, each plot is one fold. For non-tree points, which have no plot, we assign the folds at random (see Step 4).

We define the class lookup table once, here, and use it in every later cell.

In [28]:
# Class lookup table: defined once and used by all later cells
tree_species = sorted(vst_filtered['taxonID'].unique())
class_lookup = {species: idx for idx, species in enumerate(tree_species)}
class_lookup.update({'GRSS': 7, 'WATR': 8, 'SHADE': 9})
class_names = {num: name for name, num in class_lookup.items()}

# Display colors (hex) for maps
class_colors = {
    'JUVI': 'FFFF00', 'QUMA3': 'FFA500', 'QUST': '006400', 'ULCR': '800080',
    'GRSS': '90EE90', 'WATR': '0000FF', 'SHADE': '000000'
}

print('Class lookup table:')
for taxon, num in sorted(class_lookup.items(), key=lambda x: x[1]):
    print(f'  {taxon}: {num}')

Class lookup table:
  JUVI: 0
  QUMA3: 1
  QUST: 2
  ULCR: 3
  GRSS: 7
  WATR: 8
  SHADE: 9


In [29]:
# One cross-validation fold for each plot
plots_sorted = sorted(vst_filtered['plotID'].unique())
plot_to_fold = {plot: fold for fold, plot in enumerate(plots_sorted)}
fold_to_plot = {fold: plot for plot, fold in plot_to_fold.items()}
n_folds = len(plot_to_fold)

# Build tree FeatureCollection
tree_features = []
for _, row in vst_filtered.iterrows():
    geom = ee.Geometry.Point([row['longitude'], row['latitude']])
    props = {
        'taxonID': row['taxonID'],
        'class': class_lookup[row['taxonID']],
        'height': float(row['height']),
        'individualID': row['individualID'],
        'plotID': row['plotID'],
        'fold': plot_to_fold[row['plotID']]
    }
    tree_features.append(ee.Feature(geom, props))

tree_fc = ee.FeatureCollection(tree_features)
print(f'Tree FeatureCollection: {len(tree_features)} features in {n_folds} plots (folds)')

# Verify class assignments
for taxon in tree_species:
    count = tree_fc.filter(ee.Filter.eq('class', class_lookup[taxon])).size().getInfo()
    print(f'  {taxon} (class {class_lookup[taxon]}): {count} features')

Tree FeatureCollection: 63 features in 5 plots (folds)
  JUVI (class 0): 9 features
  QUMA3 (class 1): 6 features
  QUST (class 2): 41 features
  ULCR (class 3): 7 features


In [30]:
# Non-tree points have no plot. Assign them to folds at random, separately for each
# class, so that each fold gets about the same number of points of each class.
rng = np.random.default_rng(1)

def make_non_tree_fc(coords, taxon):
    folds = rng.permutation(len(coords)) % n_folds
    return ee.FeatureCollection([
        ee.Feature(ee.Geometry.Point(coord),
                   {'taxonID': taxon, 'class': class_lookup[taxon], 'fold': int(fold)})
        for coord, fold in zip(coords, folds)
    ])

grss_fc = make_non_tree_fc(grss_coords, 'GRSS')
watr_fc = make_non_tree_fc(watr_coords, 'WATR')
shade_fc = make_non_tree_fc(shade_coords, 'SHADE')

# Merge all classes
all_points = tree_fc.merge(grss_fc).merge(watr_fc).merge(shade_fc)

print(f'Total reference points: {all_points.size().getInfo()}')
print('\nPoints per class:')
for class_num in sorted(class_names):
    count = all_points.filter(ee.Filter.eq('class', class_num)).size().getInfo()
    print(f'  {class_names[class_num]} (class {class_num}): {count} points')

Total reference points: 93

Points per class:
  JUVI (class 0): 9 points
  QUMA3 (class 1): 6 points
  QUST (class 2): 41 points
  ULCR (class 3): 7 points
  GRSS (class 7): 10 points
  WATR (class 8): 10 points
  SHADE (class 9): 10 points


In [31]:
# Count points by class
class_counts = {}
for taxon, class_num in class_lookup.items():
    class_counts[taxon] = all_points.filter(ee.Filter.eq('class', class_num)).size().getInfo()

counts_df = pd.DataFrame.from_dict(class_counts, orient='index', columns=['Count'])
counts_df.index.name = 'Class'
counts_df = counts_df.sort_values('Count', ascending=False)
print('\nPoints per class:')
display(counts_df)


Points per class:


,Count
Class,
QUST,41
WATR,10
GRSS,10
SHADE,10
JUVI,9
ULCR,7
QUMA3,6


### Map 2: Reference Points by Class

Display all training points colored by class.

In [32]:
# Map 2: Reference Points
ref_points_map = geemap.Map()

ref_points_map.addLayer(
    CLBJ_SDR2022subset,
    {'bands': ['B053', 'B035', 'B019'], 'min': 0.005, 'max': 0.10},
    'RGB 2022',
    True
)

for name, class_num in class_lookup.items():
    class_points = all_points.filter(ee.Filter.eq('class', class_num))
    styled = class_points.map(lambda f: f.set('style', {
        'color': class_colors[name],
        'pointShape': 'diamond',
        'pointSize': 5,
        'width': 2,
        'fillColor': '00000000',
    }))
    ref_points_map.addLayer(styled.style(styleProperty='style'), {}, name, True)

ref_points_map.centerObject(all_points, 14)
ref_points_map

Map(center=[33.40533461408076, -97.58285335052743], controls=(WidgetControl(options=['position', 'transparent_…

## Step 4: Plan the Validation (Leave-One-Plot-Out)

We want to know how well the classifier predicts species at locations that it has not seen. A random split of stems cannot tell us this. Stems in the same plot are close together and share soil, light, and microclimate, so their spectra are similar (**spatial autocorrelation**). If stems of one plot are in both the training set and the test set, the classifier is tested next to locations that it has already seen, and the accuracy is too high.

**Leave-one-plot-out cross-validation** avoids this:
1. Each VST plot is one fold.
2. In each fold, one plot is the test set. The classifier trains on the stems of all other plots.
3. After all folds, every tree stem has been predicted by a classifier that never saw its plot. We pool the predictions into one confusion matrix.

We use this method instead of one 80/20 split of plots, because we have only a few plots. With a single test plot, the accuracy would depend on which plot we picked, and a species that occurs in only two plots could lose one of them.

**Non-tree points.** The grassland, water, and shade points have no plot. We assign them to the folds at random, with the same number of points of each class in each fold. These points are not independent in space, so the accuracy of the three non-tree classes is probably higher than it would be at new locations. We discuss this limit at the end.

The table below shows the plot that is held out in each fold and the number of tree stems of each species in that plot.

In [33]:
# Show the cross-validation design: held-out plot and tree stems in each fold
fold_rows = []
for fold, plot in fold_to_plot.items():
    counts = vst_filtered[vst_filtered['plotID'] == plot]['taxonID'].value_counts()
    row = {'Fold': fold, 'Held-out plot': plot}
    row.update({species: int(counts.get(species, 0)) for species in tree_species})
    row['Total stems'] = int(counts.sum())
    fold_rows.append(row)

fold_df = pd.DataFrame(fold_rows).set_index('Fold')
display(fold_df)

,Held-out plot,JUVI,QUMA3,QUST,ULCR,Total stems
Fold,,,,,,
0,CLBJ_002,5,0,8,4,17
1,CLBJ_003,0,1,13,0,14
2,CLBJ_004,0,0,7,0,7
3,CLBJ_006,0,4,4,0,8
4,CLBJ_043,4,1,9,3,17


### Extract Predictor Values at the Reference Points

Sample the composite image at each reference point to extract the spectral signature (381 bands) and the canopy height. We sample all points once. The `fold` property stays with each sample, so that we can split the samples by fold without sampling again.

In [34]:
# Sample the composite at all reference points (keep class and fold)
training_samples = CLBJ_composite.sampleRegions(  # Use CLBJ_composite, not _airshed
    collection=all_points,
    properties=['class', 'fold'],
    scale=1,
    tileScale=16
)

n_points = all_points.size().getInfo()
n_samples = training_samples.size().getInfo()
print(f'Reference points: {n_points}')
print(f'Samples extracted: {n_samples}')
if n_samples < n_points:
    print(f'{n_points - n_samples} points fell on masked (no-data) pixels and were dropped.')

Reference points: 93
Samples extracted: 92
1 points fell on masked (no-data) pixels and were dropped.


### Train the Random Forest Classifier

First we train one classifier on all reference points. We use it for the classified map and for the out-of-bag error. The random forest has 100 trees and uses all 382 predictor bands. We evaluate the method with cross-validation in Step 5.

In [35]:
# Train classifier
classifier = ee.Classifier.smileRandomForest(100).train(
    features=training_samples,
    classProperty='class',
    inputProperties=CLBJ_composite.bandNames()  # Match the composite used
)

print('Classifier trained')

# Classify the full composite
classified = CLBJ_composite.classify(classifier)
print('Image classified')

# Can clip the result for display if needed
classified_airshed = classified.clip(CLBJ_Airshed)

Classifier trained
Image classified


### Out-of-Bag Error

The out-of-bag (OOB) error is an error estimate that the random forest makes during training. Each tree is tested on the points that it did not use. The OOB estimate is more reliable than the resubstitution accuracy, but it **ignores the plots**: stems of the same plot are in the training and the out-of-bag samples. It is therefore too optimistic for new locations. Use the cross-validation in Step 5 as your accuracy estimate.

In [36]:
# Get OOB error from classifier explanation
classifier_info = classifier.explain().getInfo()
oob_error = classifier_info.get('outOfBagErrorEstimate', 'Not available')

if oob_error != 'Not available':
    oob_accuracy = 1 - oob_error
    print(f'Out-of-bag error: {oob_error:.4f}')
    print(f'Out-of-bag accuracy: {oob_accuracy:.4f}')
else:
    print('OOB error not available from this classifier')

Out-of-bag error: 0.2717
Out-of-bag accuracy: 0.7283


In [37]:
# Training resubstitution accuracy (the classifier is tested on its own training points)
train_classified = training_samples.classify(classifier)
train_matrix = train_classified.errorMatrix('class', 'classification')
train_accuracy = train_matrix.accuracy().getInfo()

print(f'Training resubstitution accuracy: {train_accuracy:.4f}')
print('(Note: This is optimistically biased; use the leave-one-plot-out accuracy from Step 5 instead)')

Training resubstitution accuracy: 0.9891
(Note: This is optimistically biased; use the leave-one-plot-out accuracy from Step 5 instead)


## Step 5: Leave-One-Plot-Out Cross-Validation

For each fold, we train a new random forest on the samples of all other folds and classify the samples of the held-out fold. We save the reference class and the predicted class of every held-out point. The loop trains the classifier once for each plot, so it takes longer than the single classifier above.

We report two things:
- **Accuracy for each held-out plot.** This shows how much the accuracy changes from plot to plot. Each plot has only a few stems, so these numbers are noisy.
- **Pooled accuracy.** This combines the predictions of all folds into one confusion matrix (next section).

In [38]:
# Leave-one-plot-out cross-validation
cv_frames = []
for fold in range(n_folds):
    train_k = training_samples.filter(ee.Filter.neq('fold', fold))
    test_k = training_samples.filter(ee.Filter.eq('fold', fold))

    classifier_k = ee.Classifier.smileRandomForest(100).train(
        features=train_k,
        classProperty='class',
        inputProperties=CLBJ_composite.bandNames()
    )
    predicted_k = test_k.classify(classifier_k)

    cv_frames.append(pd.DataFrame({
        'fold': fold,
        'reference': predicted_k.aggregate_array('class').getInfo(),
        'predicted': predicted_k.aggregate_array('classification').getInfo()
    }))
    print(f'Fold {fold + 1}/{n_folds} (held-out plot {fold_to_plot[fold]}): '
          f'{len(cv_frames[-1])} test points')

cv_df = pd.concat(cv_frames, ignore_index=True)
cv_df[['reference', 'predicted']] = cv_df[['reference', 'predicted']].astype(int)
cv_df['correct'] = cv_df['reference'] == cv_df['predicted']

# Accuracy for each held-out plot (tree stems) and for the non-tree points in the same fold
fold_rows = []
for fold, group in cv_df.groupby('fold'):
    trees = group[group['reference'] < 7]
    non_trees = group[group['reference'] >= 7]
    fold_rows.append({
        'Held-out plot': fold_to_plot[fold],
        'Tree stems': len(trees),
        'Tree accuracy': trees['correct'].mean(),
        'Non-tree points': len(non_trees),
        'Non-tree accuracy': non_trees['correct'].mean()
    })
print('\nAccuracy for each fold:')
display(pd.DataFrame(fold_rows).set_index('Held-out plot').style.format(
    {'Tree accuracy': '{:.2%}', 'Non-tree accuracy': '{:.2%}'}))

Fold 1/5 (held-out plot CLBJ_002): 23 test points
Fold 2/5 (held-out plot CLBJ_003): 20 test points
Fold 3/5 (held-out plot CLBJ_004): 13 test points
Fold 4/5 (held-out plot CLBJ_006): 14 test points
Fold 5/5 (held-out plot CLBJ_043): 22 test points

Accuracy for each fold:


,Tree stems,Tree accuracy,Non-tree points,Non-tree accuracy
Held-out plot,,,,
CLBJ_002,17,47.06%,6,100.00%
CLBJ_003,14,85.71%,6,100.00%
CLBJ_004,7,14.29%,6,100.00%
CLBJ_006,8,50.00%,6,83.33%
CLBJ_043,17,47.06%,5,100.00%


### Pooled Confusion Matrix and Per-Class Accuracy

The confusion matrix compares the reference class (rows) with the predicted class (columns) for all held-out points of all folds. The diagonal shows the correct predictions.

- **Producer's accuracy** is the fraction of the reference points of a class that the classifier found (the diagonal value divided by the row total). It shows how many real stems of a species the map would miss.
- **User's accuracy** is the fraction of the points that the classifier assigned to a class that really belong to this class (the diagonal value divided by the column total). It shows how far you can trust a mapped species.

In [39]:
# Pooled confusion matrix (rows = reference, columns = predicted)
labels = sorted(class_names)
label_names = [class_names[i] for i in labels]

cm = (pd.crosstab(cv_df['reference'], cv_df['predicted'])
        .reindex(index=labels, columns=labels, fill_value=0))

n_total = cm.values.sum()
overall_accuracy = np.trace(cm.values) / n_total
expected = (cm.sum(axis=1).values * cm.sum(axis=0).values).sum() / n_total**2
kappa = (overall_accuracy - expected) / (1 - expected)

confusion_df = cm.copy()
confusion_df.index = label_names
confusion_df.columns = label_names
confusion_df.index.name = 'Reference'
confusion_df.columns.name = 'Predicted'

print('Pooled confusion matrix (all folds):')
display(confusion_df)

print(f'\nHeld-out points: {n_total}')
print(f'Correctly classified: {np.trace(cm.values)}')
print(f'Overall accuracy (leave-one-plot-out): {overall_accuracy:.4f}')
print(f'Kappa coefficient: {kappa:.4f}')

Pooled confusion matrix (all folds):


Predicted,JUVI,QUMA3,QUST,ULCR,GRSS,WATR,SHADE
Reference,,,,,,,
JUVI,0,1,8,0,0,0,0
QUMA3,0,0,5,0,1,0,0
QUST,4,2,33,1,0,0,1
ULCR,0,0,7,0,0,0,0
GRSS,0,0,1,0,9,0,0
WATR,0,0,0,0,0,10,0
SHADE,0,0,0,0,0,0,9



Held-out points: 92
Correctly classified: 61
Overall accuracy (leave-one-plot-out): 0.6630
Kappa coefficient: 0.5165


In [40]:
# Per-class accuracy from the pooled confusion matrix (rows = reference, columns = predicted)
diagonal = np.diag(cm.values)
reference_totals = cm.sum(axis=1).values   # row totals
predicted_totals = cm.sum(axis=0).values   # column totals

producers_acc = np.where(reference_totals > 0, diagonal / np.maximum(reference_totals, 1), np.nan)
users_acc = np.where(predicted_totals > 0, diagonal / np.maximum(predicted_totals, 1), np.nan)

accuracy_df = pd.DataFrame({
    'Reference points': reference_totals,
    "Producer's accuracy": producers_acc,
    "User's accuracy": users_acc
}, index=label_names)

print('Per-class accuracy:')
display(accuracy_df.style.format(
    {"Producer's accuracy": '{:.2%}', "User's accuracy": '{:.2%}'}, na_rep='n/a'))

Per-class accuracy:


,Reference points,Producer's accuracy,User's accuracy
JUVI,9,0.00%,0.00%
QUMA3,6,0.00%,0.00%
QUST,41,80.49%,61.11%
ULCR,7,0.00%,0.00%
GRSS,10,90.00%,90.00%
WATR,10,100.00%,100.00%
SHADE,9,100.00%,90.00%


### How to Read These Results

Look at four things before you trust the accuracy:
1. **Compare the plots.** If the accuracy changes a lot from one held-out plot to the next, the single pooled number hides large differences between locations.
2. **Look for species that are never predicted.** A species with a producer's accuracy of 0% in the matrix is not recognized at new plots, even if the overall accuracy looks acceptable. A species with many stems, such as QUST, can dominate the overall accuracy.
3. **Compare tree classes with non-tree classes.** The non-tree points are not separated by plot, so their accuracy is probably higher than the tree accuracy for a reason that has nothing to do with the classifier.
4. **Remember the sample size.** The number of reference points in each class is small. One stem changes the accuracy of a rare species by several percentage points.

### Map 3: Classification Result

Display the classified image with training points overlaid.

In [41]:
# Map 3: Classification Results
results_map = geemap.Map()

results_map.addLayer(
    CLBJ_SDR2022subset,
    {'bands': ['B053', 'B035', 'B019'], 'min': 0.005, 'max': 0.10},
    'RGB 2022',
    True
)

# Classified image with palette. The palette needs 10 colors for the class values 0-9;
# the unused values stay grey.
palette = ['808080'] * 10
for name, class_num in class_lookup.items():
    palette[class_num] = class_colors[name]

results_map.addLayer(
    classified,
    {'min': 0, 'max': 9, 'palette': palette},
    'Classification 2022',
    True,
    0.7  # 70% opacity to see basemap through
)

# Add reference points
for name, class_num in class_lookup.items():
    class_pts = all_points.filter(ee.Filter.eq('class', class_num))
    styled = class_pts.map(lambda f: f.set('style', {'color': class_colors[name], 'pointShape': 'circle', 'pointSize': 3}))
    results_map.addLayer(styled.style(styleProperty='style'), {}, f'{name} points', False)

results_map.centerObject(CLBJ_Airshed, 13)
results_map

Map(center=[33.40123000138371, -97.57000034548355], controls=(WidgetControl(options=['position', 'transparent_…

## Discussion

### What the cross-validation measures

In leave-one-plot-out cross-validation, the classifier predicts every tree stem without having seen any stem of the same plot. The accuracy therefore estimates how well the classifier works at new plots. A random split of stems, as in Phase 2, puts stems of the same plot in the training and the test set. That split measures how well the classifier recognizes neighbors of stems that it has already seen, and it is usually higher. Compare the pooled accuracy above with the Phase 2 accuracy, and with the accuracy for each held-out plot, to see how large the difference is for these data.

### Limitations

1. **Few plots.** Each fold holds out one plot. The tree accuracy of a fold rests on the few stems of this plot. The per-plot numbers are noisy.
2. **Species in few plots.** Some species occur in only two or three plots. When one of these plots is held out, the classifier trains on the remaining plots of the species only. This makes these species hard to predict, and the estimate is uncertain.
3. **Class imbalance.** QUST has many more stems than the other species. The classifier favors common classes, and the overall accuracy is dominated by QUST. Check the producer's and user's accuracy of each class, not only the overall accuracy.
4. **Non-tree points.** The non-tree points were selected by hand from the image and are not separated by plot. Points of the same class are close together, so spatial autocorrelation remains and the accuracy of these classes is probably too high.
5. **Time difference.** The AOP flight was in late spring 2022. Some stem attributes (`plantStatus`, `canopyPosition`, `height`) come from the November 2022 bout, up to six months later. A tree that died or lost its canopy position in this time would be mislabeled in the training data.
6. **Relaxed filters.** The filters keep shaded and short stems, and some of them may not be visible from the air. The reflectance at their positions then belongs to another crown or to a gap.
7. **Position error.** The geoNEON positions have an uncertainty of about the size of a few 1 m pixels. A point can fall on the crown of a neighboring tree.
8. **No tuning.** The random forest uses the default parameters with 100 trees. We did not tune it, so the result is not the best that the data can give.
9. **Out-of-bag error.** The OOB estimate does not account for plots and is too optimistic for new locations.

## Next Steps

To extend this tutorial:

1. **Add more plots and years.** More plots give more folds and more stems for rare species. Apply the method to 2023, 2024, and 2025 at CLBJ, and match the VST bout dates to the AOP flight dates.
2. **Balance the classes.** Use oversampling, undersampling, or class weights to reduce the influence of QUST.
3. **Tune the random forest.** Test the number of trees and other parameters with a nested cross-validation, so that the tuning does not use the held-out plot.
4. **Examine the variable importance.** Find out which bands contribute most to the classification, and test whether fewer bands generalize better.
5. **Map the uncertainty.** Output the class probabilities of the random forest to show where the map is uncertain.
6. **Export the classified image** with `ee.batch.Export.image.toDrive()` for use in Geographic Information System (GIS) software.
7. **Adapt the method to other NEON sites** with different species lists.

For questions or feedback on this tutorial, contact NEON Data Skills or submit an issue to the tutorial repository.